# Bài 10 — Các Agent chuyên biệt

## Lý thuyết

### Mục tiêu

Đây là bước chuyển quan trọng: tách hệ thống thành **4 agent độc lập**, mỗi agent chỉ làm 1 việc:

| Agent | Việc làm | Công nghệ |
|---|---|---|
| Web Research Agent | Tìm tin tức mới nhất về cổ phiếu | DuckDuckGo search |
| Financial Data Agent | Lấy giá, chỉ số tài chính | yfinance (đã làm ở Phase 1) |
| RAG Agent | Trả lời câu hỏi từ báo cáo PDF | ChromaDB + RAG (đã làm ở Phase 2) |
| Analysis Agent | Tổng hợp dữ liệu, phân tích xu hướng | LLM thuần (không cần tool) |

### Tại sao tách thành nhiều agent?

Một agent "làm tất cả" sẽ có prompt quá dài, khó kiểm soát, khó debug khi sai. Tách nhỏ giúp:

- Mỗi agent có **system prompt riêng**, tập trung 1 việc → chất lượng cao hơn
- Dễ test độc lập từng agent
- Dễ thay thế 1 agent mà không ảnh hưởng agent khác

### Agent trong LangGraph là gì?

Ở Bài 9, mỗi **node** chỉ làm 1 bước xử lý đơn giản. Từ bài này, mỗi **node sẽ chính là 1 agent** — vẫn là hàm Python nhận state, trả state, nhưng bên trong nó có thể gọi LLM, gọi tool, làm logic phức tạp hơn.

```
State đi qua:
  Web Research Agent (node) → Financial Data Agent (node) → RAG Agent (node) → Analysis Agent (node)
```

Ở Bài 10 này, ta xây **từng agent độc lập, test riêng lẻ** — chưa nối thành graph (việc đó là Bài 11, Orchestrator).

### Kiến trúc tổng thể của Bài 10

```
                ┌─────────────────────────┐
                │   Câu hỏi / Mã cổ phiếu  │
                └────────────┬─────────────┘
                              │
        ┌──────────────┬─────┴───────┬──────────────┐
        ▼              ▼             ▼              │
┌───────────────┐┌────────────┐┌───────────┐         │
│ Web Research   ││ Financial  ││    RAG    │         │
│ Agent          ││ Data Agent ││   Agent   │         │
│ (tin tức mới)  ││ (giá, P/E) ││ (PDF nội  │         │
│                ││            ││  dung)    │         │
└───────┬────────┘└─────┬──────┘└─────┬─────┘         │
        │                │             │              │
        └────────────────┴─────────────┘              │
                          │                            │
                          ▼                            │
                ┌──────────────────┐                   │
                │  Analysis Agent   │ ◄─────────────────┘
                │  (tổng hợp +      │
                │   phân tích)      │
                └─────────┬─────────┘
                          ▼
                  Kết quả phân tích
```

Mỗi agent **độc lập, không phụ thuộc nhau** ở bước này — chỉ Analysis Agent cần dữ liệu từ 3 agent kia. Việc điều phối thứ tự chạy (ai chạy trước, ai chạy sau, có cần chạy không) sẽ do **Orchestrator** quyết định ở Bài 11.

### Pattern chung của 1 Agent

Nhìn lại agent đã làm ở Bài 8 (`query_financial_report`), ta sẽ thấy mọi agent đều theo cùng 1 khuôn:

```
1. Lấy dữ liệu thật   (web search / API / vector database)
2. Đưa cho LLM         (đóng gói dữ liệu vào prompt)
3. Trả kết quả         (LLM tổng hợp thành câu trả lời tự nhiên)
```

Khác biệt giữa các agent chỉ là **bước 1 — nguồn dữ liệu**. Đây là lý do gọi chúng là "chuyên biệt" (specialized) — mỗi agent chuyên 1 nguồn dữ liệu duy nhất.

### Thư viện mới: DuckDuckGo Search

Dùng để tìm tin tức thật trên web, miễn phí, không cần API key.

```bash
pip install ddgs
```

| Hàm | Ý nghĩa |
|---|---|
| `DDGS()` | Tạo 1 session tìm kiếm DuckDuckGo |
| `ddgs.text(query, max_results=3)` | Tìm kiếm theo `query`, trả về list dict, mỗi dict có `title`, `body`, `href` |

## Ví dụ — Web Research Agent

Agent này: tìm tin tức thật → đưa LLM tóm tắt → trả về văn bản tóm tắt.

In [1]:
import os
from dotenv import load_dotenv
from google import genai
from ddgs import DDGS

load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

In [2]:
def web_research_agent(query):
    # Bước 1: Tìm kiếm tin tức thật
    with DDGS() as ddgs:
        results = ddgs.text(query, max_results=3)

    # Bước 2: Ghép các kết quả tìm được thành context
    news_context = "\n\n".join([f"{r['title']}: {r['body']}" for r in results])

    # Bước 3: Đưa cho LLM tổng hợp thành câu trả lời tự nhiên
    prompt = f"""Dựa vào các tin tức sau:

{news_context}

Hãy tóm tắt thông tin quan trọng nhất."""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text

In [3]:
summary = web_research_agent("NVIDIA stock news latest")
print(summary)

Dựa trên các tin tức được cung cấp, thông tin quan trọng nhất có thể tóm tắt như sau:

1.  **NVIDIA là tâm điểm chú ý nhưng có góc nhìn đa chiều:** Cổ phiếu NVIDIA (NVDA) tiếp tục là chủ đề được thảo luận rộng rãi trên các kênh tài chính lớn, cho thấy sự quan tâm cao của thị trường và giới đầu tư. Tuy nhiên, Jim Cramer của CNBC lại bày tỏ sự yêu thích một cổ phiếu công nghệ truyền thống khác hơn NVIDIA, ngụ ý rằng có những lựa chọn thay thế hoặc góc nhìn khác về tiềm năng đầu tư.
2.  **SpaceX được đánh giá có tiềm năng tăng trưởng khổng lồ:** Michael Maguire của Sequoia Capital đã so sánh SpaceX với NVIDIA của ba năm trước. Đây là một nhận định rất lạc quan, ngụ ý rằng SpaceX đang đứng trước một giai đoạn tăng trưởng bùng nổ tương tự như NVIDIA đã trải qua.
3.  **Các công ty AI (Anthropic, OpenAI) cũng là chủ đề nóng:** Anthropic và OpenAI được nhắc đến cùng với NVIDIA và SpaceX trong các cuộc thảo luận trên truyền thông tài chính, cho thấy tầm quan trọng ngày càng tăng của các công ty

Nhận thấy gì? Cấu trúc **giống hệt** `query_financial_report` ở Bài 8 — chỉ khác nguồn dữ liệu (web search thay vì ChromaDB). Đây chính là pattern chung: **Lấy dữ liệu → Đưa LLM tổng hợp → Trả kết quả**.

## Bài tập

Viết và test **4 agent độc lập** (chưa cần ghép vào LangGraph):

1. `web_research_agent(query)` — đã có sẵn ở trên, dùng lại được
2. `financial_data_agent(symbol)` — tái sử dụng `get_stock_price` từ Phase 1, nhưng mở rộng thêm: lấy cả `marketCap`, P/E ratio (`trailingPE`) từ `ticker.info`, trả về **dict** đầy đủ
3. `rag_agent(question)` — tái sử dụng `query_financial_report` từ Bài 8 (copy lại hàm, dùng ChromaDB đã lưu ở Bài 6)
4. `analysis_agent(data)` — nhận 1 đoạn text tổng hợp (gộp kết quả 3 agent trên), yêu cầu LLM phân tích xu hướng, không cần tool gì cả — chỉ 1 lời gọi LLM với prompt phân tích

Test từng agent riêng với input về NVIDIA, in kết quả ra để kiểm tra từng agent hoạt động đúng trước khi ghép lại ở Bài 11.

Viết code của bạn vào các ô dưới đây:

In [4]:
import yfinance as yf

# 1. financial_data_agent
def financial_data_agent(symbol):
    ticker_obj = yf.Ticker(symbol)
    info = ticker_obj.info
    return {
        "symbol": symbol,
        "currentPrice": info.get("currentPrice", "Không tìm thấy giá cổ phiếu"),
        "marketCap": info.get("marketCap", "Không tìm thấy marketCap"),
        "trailingPE": info.get("trailingPE", "Không tìm thấy trailingPE")
    }

In [9]:
# 2. rag_agent
from sentence_transformers import SentenceTransformer
import chromadb

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
db_client = chromadb.PersistentClient(path="D:\Code\AI\Multi-Agent Research Assistant project\phase2\chroma_db")
collection = db_client.get_or_create_collection(name="nvidia_report")

def rag_agent(question):
    question_embedding = embed_model.encode(question)
    results = collection.query(
        query_embeddings=[question_embedding.tolist()],
        n_results=3
    )
    chunks = results["documents"][0]
    context = "\n\n".join(chunks)

    prompt = f"""Dựa vào thông tin sau từ báo cáo tài chính NVIDIA:

{context}

Hãy trả lời câu hỏi: {question}
Nếu thông tin không đủ để trả lời, hãy nói rõ là không tìm thấy thông tin liên quan.
    """

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text

<>:6: SyntaxWarning: invalid escape sequence '\C'
<>:6: SyntaxWarning: invalid escape sequence '\C'
C:\Users\hhoan\AppData\Local\Temp\ipykernel_15584\1783101472.py:6: SyntaxWarning: invalid escape sequence '\C'
  db_client = chromadb.PersistentClient(path="D:\Code\AI\Multi-Agent Research Assistant project\phase2\chroma_db")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3611.22it/s]


In [12]:
# 3. analysis_agent
def analysis_agent(data):
    prompt = f"""Dựa vào thông tin tổng hợp sau về tài chính NVIDIA:

{data}

Hãy tổng hợp thông tin, phân tích xu hướng. 
Chỉ sử dụng dữ liệu được cung cấp ở trên. Không tự thêm số liệu, sự kiện hay bất kì thông tin nào không có trong dữ liệu này.
    """
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text

In [ ]:
# 4. Test cả 4 agent với NVIDIA

# test web research agent
news_summary = web_research_agent("NVIDIA stock news latest")
print("Web Research Agent: ")
print(news_summary)

# test financial data agent
financial_data = financial_data_agent("NVDA")
print("\nFianancial Data Agent:")
print(financial_data)

# test RAG Agent
rag_result = rag_agent("What was NVIDIA's revenue in Q1 2027?")
print("\n RAG Agent:")
print(rag_result)

# Test Analyst Agent - gộp kết quả 3 agent trên thành 1 đoạn rồi đưa cho LLM phân tích
combined_data = f"""
Tin tức : {news_summary}

Dữ liệu tài chính hiện tại: {financial_data}

Báo cáo: {rag_result}
"""

analysis_result = analysis_agent(combined_data)
print("\n Analysis Agent: ")
print(analysis_result)

Web Research Agent: 
Dựa trên các tin tức được cung cấp, đây là những thông tin quan trọng nhất:

1.  **NVIDIA (NVDA) là tâm điểm thảo luận:** Cổ phiếu NVIDIA tiếp tục là chủ đề nóng được các chuyên gia từ CNBC, Bloomberg, Yahoo Finance và Schwab Network bàn luận rộng rãi, cho thấy sự quan tâm lớn của thị trường.
2.  **Các công ty công nghệ lớn khác cũng được chú ý:** Anthropic, SpaceX và OpenAI cũng được đưa vào các cuộc thảo luận cùng với NVIDIA, cho thấy tầm quan trọng của chúng trong bối cảnh công nghệ và tài chính hiện tại.
3.  **Quan điểm trái chiều về NVIDIA:** Jim Cramer của CNBC bày tỏ sự yêu thích một "cổ phiếu công nghệ truyền thống" hơn NVIDIA, cho thấy không phải tất cả các chuyên gia đều đặt NVIDIA lên hàng đầu so với mọi lựa chọn khác.
4.  **SpaceX được đánh giá cao:** Maguire của Sequoia đã so sánh SpaceX với NVIDIA của ba năm trước, ngụ ý rằng SpaceX có tiềm năng tăng trưởng bùng nổ tương tự như NVIDIA đã trải qua trong giai đoạn gần đây. Điều này cho thấy kỳ vọng rất 